# UDF in Spark

## Understanding UDF in PySpark

In PySpark, a User Defined Function (UDF) is a custom function written in Python (or other supported languages) that can be applied to DataFrame columns. It allows users to perform transformations and computations that are not natively supported by Spark's built-in functions.

UDFs bridge the gap between Spark's optimized, distributed engine and user-defined logic. However, UDFs often come with performance trade-offs because they operate outside Spark's optimization framework.

**Key Points About UDFs:**
- UDFs are used for column-wise transformations.
- The UDF logic is applied row by row, and UDFs run on Spark's worker nodes.
- UDFs are slower than Spark SQL or DataFrame built-in functions since they don't take advantage of Spark's Catalyst optimizer.

## Lab: Validate IP Address

### Step 1: Initialize SparkSession

In [1]:
%pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import BooleanType

spark = SparkSession.builder \
    .appName("UDF Lab - Validate IP Address") \
    .getOrCreate()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/15 14:00:59 WARN Utils: Your hostname, Vijays-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.25.37.51 instead (on interface en0)
26/09/15 14:00:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 14:00:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/15 14:01:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Step 2: Define the Python function

A plain Python function that checks whether a given IP address string is valid.

In [2]:
def is_valid_ip(ip_str):
    """
    Function to check if the given IP address string is valid or not.
    """
    try:
        # Split the string by '.' to get parts of the IP address
        parts = ip_str.split('.')

        # Check if there are exactly 4 parts
        if len(parts) != 4:
            return False

        # Check each part is a valid number between 0 and 255
        for part in parts:
            if not part.isdigit():  # Ensure each part is a number
                return False
            if not (0 <= int(part) <= 255):  # Ensure each number is in the range 0-255
                return False

        return True  # IP is valid
    except Exception:
        return False

### Step 3: Register the UDF

In [3]:
is_valid_ip_udf = udf(is_valid_ip, BooleanType())

/Users/vijay/Desktop/Ness/workspace/ness_datalake/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


### Step 4: Create Sample Data

In [4]:
data = [
    ("192.168.1.1",),
    ("10.0.0.256",),
    ("172.16.300.1",),
    ("abc.def.ghi.jkl",),
    ("255.255.255.255",),
    ("0.0.0.0",),
    ("123.045.067.089",),
]

columns = ["ip_address"]

# Create a DataFrame
df = spark.createDataFrame(data, columns)

### Step 5: Apply the UDF to the DataFrame

In [5]:
result_df = df.withColumn("is_valid", is_valid_ip_udf(col("ip_address")))

# Show the Results
result_df.show(truncate=False)

+---------------+--------+
|ip_address     |is_valid|
+---------------+--------+
|192.168.1.1    |true    |
|10.0.0.256     |false   |
|172.16.300.1   |false   |
|abc.def.ghi.jkl|false   |
|255.255.255.255|true    |
|0.0.0.0        |true    |
|123.045.067.089|true    |
+---------------+--------+

